In [ ]:
import pandas as pd
data=pd.read_csv('titanic.csv')
data

In [ ]:
#Work on a copy
df=data.copy()

In [ ]:
#exploring data
df.head()

In [51]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   PassengerId    891 non-null    int64  
 1   Survived       891 non-null    int64  
 2   Pclass         891 non-null    int64  
 3   Name           891 non-null    str    
 4   Sex            891 non-null    str    
 5   Age            891 non-null    float64
 6   SibSp          891 non-null    int64  
 7   Parch          891 non-null    int64  
 8   Ticket         891 non-null    str    
 9   Fare           891 non-null    float64
 10  Cabin          204 non-null    str    
 11  Embarked       889 non-null    str    
 12  Deck           891 non-null    str    
 13  Cabin_missing  891 non-null    bool   
dtypes: bool(1), float64(2), int64(5), str(6)
memory usage: 91.5 KB


age, cabin and embarked have missing values

In [ ]:
df.describe()

Only 38% survived

In [ ]:
#check on missing values
df.isnull().sum()

In [ ]:
df.columns


In [ ]:
# checking unique values in each column
sur=df['Survived'].unique()
print(sur,"\n")
pc=df['Pclass'].unique()
print(pc,'\n')
sex=df['Sex'].unique()
print(sex,'\n')
sib=df['SibSp'].unique()
print(sib,'\n')
tic=df['Ticket'].unique()
print(tic,'\n')
cab=df['Cabin'].unique()
print(cab,'\n')
emb=df['Embarked'].unique()
print(emb,'\n')

Start cleaning data:-

In [ ]:
#filling the two missed embarked values with the most frequent value (mfv)
mfv=df['Embarked'].mode()[0]
df['Embarked'].fillna(mfv,inplace=True)

Working on Age:
1- Understanding the data and its defects

In [ ]:
age_stats = df.groupby('Pclass')['Age'].agg(['min', 'max', 'mean', 'median', 'count', 'size'])
age_stats


In [ ]:
age_stats['range']=age_stats['max']-age_stats['min']
age_stats


In [ ]:
#calculating total missing ages
total_missing_ages =df['Age'].isna().sum()
total_missing_ages

In [ ]:
#calculate total missing Ages (tma) for each class op passengers
tma=df.groupby('Pclass')['Age'].apply(lambda x: x.isna().sum())
tma
#calculate the percentage of missing ages for each
total_age_by_class=df.groupby(['Pclass'])['Age'].size()
total_age_by_class
tma_percentage= (tma/total_age_by_class)*100
tma_percentage


Filling the missing ages by the mean value of each class:-


In [ ]:
df['Age']=df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.mean()))
df['Age']

All age entries are filled


Exploratory Data Analysis (EDA)
understanding the relation between survival rates and other factors.
Working on cabin data:-

In [ ]:
# Extract deck letter
df['deck'] = df['Cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'Missing')
# Calculate missing cabin info by class
missing_by_class = df.groupby('Pclass')['Cabin'].apply(lambda x: f"{x.isna().sum()}/{len(x)} ({x.isna().mean()*100:.1f}%)")
missing_by_class


Missing cabin data increases by class.
Cabin recording was highly biased toward wealthy passengers.

In [ ]:
# Deck distribution by class (excluding missing)
known = df[df['deck'] != 'Missing']
deck_by_class = pd.crosstab(known['Pclass'], known['deck'])
deck_by_class


 First class passengers occupied upper decks A-E, while second class was on decks D-F, and third class was only on lower decks F-G.

In [ ]:
# Survival by class and deck
survival = df.groupby(['Pclass', 'deck'])['Survival'].mean().unstack() * 100
print(survival.round(1))

Survival rates were highest on upper decks (B: 67%, C: 71%, D: 67%), while passengers with missing cabin info had lower survival rates (21-43%).

working on gender factor:-

In [ ]:
sex_survival = df.groupby('Sex')['Survival'].value_counts(normalize=True).unstack()
print(sex_survival)
for sex in ['female', 'male']:
    rate = df[df['sex'] == sex]['survival'].value_counts(normalize=True)[1] * 100
    print(f"{sex.capitalize()}: {rate:.2f}% survived")

Females had ~74% survival, males only ~19%.